In [1]:
%pip install sacrebleu 
%pip install rouge-score
%pip install sentence-transformers
%pip install numpy

  Using cached sacrebleu-2.5.1-py3-none-any.whl.metadata (51 kB)
  Using cached portalocker-3.2.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached regex-2025.11.3-cp313-cp313-win_amd64.whl.metadata (41 kB)
  Using cached tabulate-0.9.0-py3-none-any.whl.metadata (34 kB)
  Using cached lxml-6.0.2-cp313-cp313-win_amd64.whl.metadata (3.7 kB)
Using cached sacrebleu-2.5.1-py3-none-any.whl (104 kB)
Using cached tabulate-0.9.0-py3-none-any.whl (35 kB)
Using cached lxml-6.0.2-cp313-cp313-win_amd64.whl (4.0 MB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached nltk-3.9.2-py3-none-any.whl.metadata (3.2 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.2-py3-none-any.whl (1.5 MB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=25027 sha256=3cfcfd015c190d6f930094014988790f6095ceaf42345ffb1496196dd3a334ae
  Stored in directory: c:\users\sahil\appdata\local\pip\cache\wheels\44\af\da\5ffc433e2786f0b1a9c6f458d5fb8f611d8eb332387f18698f
Successfully built rouge-score
Note: you may ne


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached sentence_transformers-5.1.2-py3-none-any.whl.metadata (16 kB)
  Using cached transformers-4.57.3-py3-none-any.whl.metadata (43 kB)
  Using cached torch-2.9.1-cp313-cp313-win_amd64.whl.metadata (30 kB)
  Using cached huggingface_hub-1.1.6-py3-none-any.whl.metadata (13 kB)
  Using cached filelock-3.20.0-py3-none-any.whl.metadata (2.1 kB)
  Using cached fsspec-2025.10.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-win_amd64.whl.metadata (5.0 kB)
  Using cached shellingham-1.5.4-py2.py3-none-any.whl.metadata (3.5 kB)
  Using cached typer_slim-0.20.0-py3-none-any.whl.metadata (16 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6-py3-none-any.whl.metadata (6.8 kB)
  Using cached setuptools-80.9.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached huggingface_hub-0.36.0-py3-none-any.whl.metadata (14 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-win_amd64.whl.metadata (6.9 kB)
  Using cached safetensors-0.7.0


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer
import sacrebleu

c:\Users\SAHIL\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_csv("G:\\Resume-Matcher\\experiments\\prompts\\results\\responses.csv")
df.head()  

,id,candidate,job,ground_truth,zero_text,few_text,adv_text,zero_label,few_label,adv_label
0,1,"Experienced Python developer, ML projects, sci...",ML Engineer,High,Medium\nThe candidate has relevant experience ...,High — extensive ML background matches the rol...,Here's a step-by-step assessment of the candid...,Medium,High,High
1,2,"Frontend React developer, Redux, CSS, 2 years",Backend Node.js Developer,Low,(Medium)\nThe candidate's skills in CSS and Re...,Low — lacks backend/Node.js experience require...,Step 1: Frontend experience vs Backend job req...,Medium,Low,Medium
2,3,"Data analyst with SQL, Tableau, Excel",Business Analyst,Medium,(Medium)\n\nThe candidate's data analysis skil...,Higher - candidate has a broader set of busine...,Here's a step-by-step analysis of the candidat...,Medium,High,High
3,4,"Junior data scientist, Python, SQL, internship...",Data Analyst,High,Medium\nThe candidate has relevant technical s...,Low — lacking relevant work experience and may...,**Step 1: Candidate's relevant skills**\n- The...,Medium,Low,Medium
4,5,"DevOps engineer, Docker, Kubernetes, AWS",Site Reliability Engineer,High,Medium\nThe candidate has relevant skills in D...,"High — aligns with SRE duties, requiring DevOp...",To assess this candidate for the Site Reliabil...,Medium,High,Medium


In [5]:
def bleu_score(pred, ref):
    return sacrebleu.sentence_bleu(pred, [ref]).score

rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
def rougeL_score(pred, ref):
    return rouge.score(ref, pred)['rougeL'].fmeasure

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

def semantic_sim(pred, ref):
    e1 = embedder.encode(pred, convert_to_tensor=True)
    e2 = embedder.encode(ref, convert_to_tensor=True)
    return float(util.cos_sim(e1, e2))


c:\Users\SAHIL\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SAHIL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not insta

In [7]:
def evaluate_row(row):
    ref = row["ground_truth"]

    # Zero-shot
    bleu_z = bleu_score(row["zero_text"], ref)
    rouge_z = rougeL_score(row["zero_text"], ref)
    sem_z = semantic_sim(row["zero_text"], ref)

    # Few-shot
    bleu_f = bleu_score(row["few_text"], ref)
    rouge_f = rougeL_score(row["few_text"], ref)
    sem_f = semantic_sim(row["few_text"], ref)

    # Advanced
    bleu_a = bleu_score(row["adv_text"], ref)
    rouge_a = rougeL_score(row["adv_text"], ref)
    sem_a = semantic_sim(row["adv_text"], ref)

    return pd.Series({
        "bleu_zero": bleu_z, "rouge_zero": rouge_z, "semantic_zero": sem_z,
        "bleu_few": bleu_f, "rouge_few": rouge_f, "semantic_few": sem_f,
        "bleu_adv": bleu_a, "rouge_adv": rouge_a, "semantic_adv": sem_a,
    })

results = df.apply(evaluate_row, axis=1)
final_df = pd.concat([df, results], axis=1)
final_df.to_csv("G:\\Resume-Matcher\\experiments\\prompts\\results\\evaluation_results.csv", index=False)
final_df.head()

,id,candidate,job,ground_truth,zero_text,few_text,adv_text,zero_label,few_label,adv_label,bleu_zero,rouge_zero,semantic_zero,bleu_few,rouge_few,semantic_few,bleu_adv,rouge_adv,semantic_adv
0,1,"Experienced Python developer, ML projects, sci...",ML Engineer,High,Medium\nThe candidate has relevant experience ...,High — extensive ML background matches the rol...,Here's a step-by-step assessment of the candid...,Medium,High,High,0.0000,0.0000,0.053917,4.196115,0.222222,0.193548,0.125153,0.008547,0.071031
1,2,"Frontend React developer, Redux, CSS, 2 years",Backend Node.js Developer,Low,(Medium)\nThe candidate's skills in CSS and Re...,Low — lacks backend/Node.js experience require...,Step 1: Frontend experience vs Backend job req...,Medium,Low,Medium,0.0000,0.0000,0.044985,2.839839,0.181818,0.375428,0.160344,0.011696,0.069601
2,3,"Data analyst with SQL, Tableau, Excel",Business Analyst,Medium,(Medium)\n\nThe candidate's data analysis skil...,Higher - candidate has a broader set of busine...,Here's a step-by-step analysis of the candidat...,Medium,High,High,1.1231,0.0625,0.062332,0.000000,0.000000,0.136310,0.116878,0.007663,0.109229
3,4,"Junior data scientist, Python, SQL, internship...",Data Analyst,High,Medium\nThe candidate has relevant technical s...,Low — lacking relevant work experience and may...,**Step 1: Candidate's relevant skills**\n- The...,Medium,Low,Medium,0.0000,0.0000,-0.042943,0.000000,0.000000,0.161115,0.000000,0.000000,0.030700
4,5,"DevOps engineer, Docker, Kubernetes, AWS",Site Reliability Engineer,High,Medium\nThe candidate has relevant skills in D...,"High — aligns with SRE duties, requiring DevOp...",To assess this candidate for the Site Reliabil...,Medium,High,Medium,0.0000,0.0000,0.028517,2.839839,0.166667,0.257208,0.000000,0.000000,0.027589
